**Classification du Churn Bancaire - Modele Custom vs Fonction Native dans Snowflake**

*Version 2 : Optimisation avec dataset 165k clients (Kaggle - Bank Customer Churn)*

## 1. Contexte

#### Problematique
Une banque souhaite **identifier en avance les clients susceptibles de partir** (churn).
Detecter ces clients permet de lancer des actions de retention ciblees avant qu'il ne soit trop tard.

### Pourquoi XGBoost ?
XGBoost (Extreme Gradient Boosting) est particulierement adapte a ce cas d'usage :

| Critere | XGBoost | Regression logistique | Random Forest |
|---|---|---|---|
| Gestion du desequilibre de classes | Excellent | Limite | Possible |
| Performance sur donnees tabulaires | Excellent | Moyen | Bon |
| Importance des features | Native | Non | Possible |
| Integration Snowflake ML Registry | Supporte | Supporte | Supporte |
| Temps d'entrainement | Rapide | Tres rapide | Plus lent |

XGBoost construit des arbres de decision en sequence, chaque arbre corrigeant les erreurs du precedent.

### Objectif
- **Variable cible :** `EXITED` (0 = client stable, 1 = client parti)
- **Dataset :** 165 034 clients, 11 features
- **Amelioration v2 :** Passage de 210 a 165k clients pour une meilleure generalisation

## 2. Source des donnees

#### Architecture de la base Snowflake
```
ML_CHURN_PROJECT (database)
   DATA_RAW (schema)
        DATASET_100K     - dataset Kaggle Bank Customer Churn (165k lignes)
   FEATURE_STORE (schema)
        CLIENT_FEATURES  - vue avec les features selectionnees (11 colonnes)
   ML_MODELS (schema)
        CHURN_PREDICTOR  - modele XGBoost dans le ML Registry
```

#### Features retenues
| Feature | Description |
|---------|-------------|
| CREDITSCORE | Score de credit du client |
| AGE | Age du client |
| TENURE | Anciennete (annees) |
| BALANCE | Solde du compte |
| NUMOFPRODUCTS | Nombre de produits bancaires |
| HASCRCARD | Possede une carte de credit (0/1) |
| ISACTIVEMEMBER | Membre actif (0/1) |
| ESTIMATEDSALARY | Salaire estime |
| GEOGRAPHY_GERMANY | Client en Allemagne (0/1) |
| GEOGRAPHY_SPAIN | Client en Espagne (0/1) |
| GENDER_MALE | Genre masculin (0/1) |

In [ ]:
%%sql -r vue_features
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ML_WH;
USE DATABASE ML_CHURN_PROJECT;
USE SCHEMA FEATURE_STORE;

CREATE OR REPLACE VIEW CLIENT_FEATURES AS
SELECT
    CREDITSCORE,
    AGE,
    TENURE,
    BALANCE,
    NUMOFPRODUCTS,
    HASCRCARD,
    ISACTIVEMEMBER,
    ESTIMATEDSALARY,
    GEOGRAPHY_GERMANY,
    GEOGRAPHY_SPAIN,
    GENDER_MALE,
    EXITED
FROM ML_CHURN_PROJECT.DATA_RAW.DATASET_100K;

SELECT COUNT(*) AS NB_LIGNES, SUM(EXITED) AS NB_CHURNERS FROM CLIENT_FEATURES;

## 3. Methode

### Strategie de split

| | XGBoost Custom | Classification Native |
|--|--|--|
| **Split** | 80% train / 20% test | ~98.5% train / ~1.5% eval (automatique) |
| **Taille train** | 132 027 clients | ~162 671 clients |
| **Taille test/eval** | 33 007 clients | ~2 363 clients |
| **Controle** | `random_state=42` (reproductible) | Split interne (non configurable) |

Le split 80/20 de XGBoost est standard en ML. La classification native utilise un split beaucoup plus agressif (quasi toutes les donnees en training), ce qui peut donner un leger avantage d'apprentissage mais rend l'evaluation moins robuste.

### Pipeline - Approche 1 : XGBoost (Python + Snowpark)
```
Snowflake (DATA_RAW.DATASET_100K)
     |
  Feature Store (VIEW CLIENT_FEATURES)
     |
  Snowpark -> Pandas
     |
  Train / Test Split (80/20)
     |
  XGBoost Classifier (scale_pos_weight pour desequilibre)
     |
  Evaluation (confusion matrix, classification report)
     |
  Snowflake ML Registry (CHURN_PREDICTOR V2)
```

**Commandes utiles :**
- `registry.log_model()` : sauvegarder le modele
- `model_ref.run(df, function_name="predict")` : scorer de nouvelles donnees
- `registry.get_model("CHURN_PREDICTOR").version("V2")` : recuperer le modele

### Pipeline - Approche 2 : Classification Native Snowflake
```
Snowflake (DATA_RAW.DATASET_100K)
     |
  Feature Store (VIEW CLIENT_FEATURES)
     |
  SNOWFLAKE.ML.CLASSIFICATION (target = EXITED)
     |
  Split interne automatique (~98.5/1.5)
     |
  Evaluation (metriques automatiques)
     |
  Predictions via PREDICT()
```

**Commandes utiles :**
- `CREATE SNOWFLAKE.ML.CLASSIFICATION model_name(...)` : creer le modele
- `CALL model_name!SHOW_EVALUATION_METRICS()` : precision, recall, f1
- `CALL model_name!SHOW_GLOBAL_EVALUATION_METRICS()` : AUC
- `CALL model_name!SHOW_CONFUSION_MATRIX()` : matrice de confusion
- `CALL model_name!SHOW_FEATURE_IMPORTANCE()` : importance des features
- `CALL model_name!SHOW_TRAINING_LOGS()` : logs d'entrainement
- `model_name!PREDICT(OBJECT_CONSTRUCT(*))` : predire sur une ligne

### Specificites de chaque approche

| Critere | XGBoost Custom | Classification Native |
|---------|----------------|----------------------|
| Langage | Python (Snowpark) | SQL uniquement |
| Controle hyperparametres | Total (n_estimators, max_depth, lr...) | Aucun (boite noire) |
| Gestion desequilibre | `scale_pos_weight` configurable | Automatique interne |
| Feature engineering | Manuel (choix des colonnes) | Automatique (toutes colonnes sauf cible) |
| Deploiement | ML Registry -> PREDICT en SQL | Directement via PREDICT() |
| Temps de mise en place | ~30 min de code | ~5 min (3 lignes SQL) |
| Reproductibilite | `random_state` fixe | Non garanti |

## 4. Implementation

### 4.1 Approche 1 : XGBoost (Python)


In [ ]:
from snowflake.snowpark.context import get_active_session
from sklearn.model_selection import train_test_split
import xgboost as xgb

session = get_active_session()
session.sql("use database ml_churn_project").collect()
session.sql("use schema ml_models").collect()
session.sql("use warehouse ml_wh").collect()

print("Base :", session.get_current_database())
print("Schema :", session.get_current_schema())
print("Warehouse :", session.get_current_warehouse())

In [ ]:
df = session.table("ML_CHURN_PROJECT.FEATURE_STORE.CLIENT_FEATURES")
df_pandas = df.to_pandas()
print(f"Nombre de clients : {len(df_pandas)}")
print(f"Repartition : {df_pandas['EXITED'].value_counts().to_dict()}")
df_pandas.head(5)

### Hyperparametres XGBoost retenus
| Parametre | Valeur | Justification |
|---|---|---|
| `n_estimators` | 1000 | Plus d'arbres pour capter des patterns complexes sur 165k lignes |
| `max_depth` | 7 | Profondeur suffisante sans surapprentissage |
| `learning_rate` | 0.1 | Standard pour XGBoost |
| `eval_metric` | logloss | Adapte a la classification binaire |
| `scale_pos_weight` | ~3.7 | Compense le desequilibre (79% reste vs 21% parti) |

In [ ]:
import pandas as pd

features = [
    "CREDITSCORE",
    "AGE",
    "TENURE",
    "BALANCE",
    "NUMOFPRODUCTS",
    "HASCRCARD",
    "ISACTIVEMEMBER",
    "ESTIMATEDSALARY",
    "GEOGRAPHY_GERMANY",
    "GEOGRAPHY_SPAIN",
    "GENDER_MALE"
]
X = df_pandas[features]
y = df_pandas["EXITED"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train : {len(X_train)} clients | Test : {len(X_test)} clients")

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight : {scale_pos:.2f}")

modele = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.1,
    scale_pos_weight=scale_pos,
    random_state=42,
    eval_metric="logloss"
)
modele.fit(X_train, y_train)

score = modele.score(X_test, y_test)
print(f"Accuracy sur le test : {score:.1%}")

importance = pd.DataFrame({
    "feature": features,
    "importance": modele.feature_importances_
}).sort_values("importance", ascending=False)
print("\nImportance des features :")
print(importance.to_string(index=False))

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="ML_CHURN_PROJECT",
    schema_name="ML_MODELS"
)

try:
    registry.delete_model("CHURN_PREDICTOR")
    print("Ancien modele supprime")
except:
    print("Pas d'ancien modele a supprimer")

model_ref = registry.log_model(
    model=modele,
    model_name="CHURN_PREDICTOR",
    version_name="V2",
    sample_input_data=X_train,
    target_platforms=["WAREHOUSE"],
    comment="XGBoost churn bancaire v2 - 165k clients"
)
print("Modele sauvegarde dans le registry (V2)")

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = modele.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("Matrice de Confusion")
print(f"  Vrais Negatifs  (reste -> predit reste) : {cm[0][0]}")
print(f"  Faux Positifs   (reste -> predit parti) : {cm[0][1]}")
print(f"  Faux Negatifs   (parti -> predit reste) : {cm[1][0]}")
print(f"  Vrais Positifs  (parti -> predit parti) : {cm[1][1]}")

print("\nRapport de Classification")
print(classification_report(y_test, y_pred, target_names=["Reste (0)", "Parti (1)"]))

### 4.2 Approche 2 : Fonction native de classification Snowflake

In [ ]:
%%sql -r dataframe_4
-----------------------------------------------------------
-- SETUP
-----------------------------------------------------------
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE ML_WH;
USE DATABASE ML_CHURN_PROJECT;
USE SCHEMA FEATURE_STORE;

-----------------------------------------------------------
-- CLASSIFICATION NATIVE
-----------------------------------------------------------
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION my_model_100k(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'CLIENT_FEATURES'),
    TARGET_COLNAME => 'EXITED',
    CONFIG_OBJECT => { 'ON_ERROR': 'SKIP' }
);

CALL my_model_100k!SHOW_TRAINING_LOGS();

CREATE OR REPLACE TABLE My_classification_v2 AS SELECT
    *,
    my_model_100k!PREDICT(
        OBJECT_CONSTRUCT(*),
        {'ON_ERROR': 'SKIP'}
    ) as predictions
FROM CLIENT_FEATURES;

SELECT * FROM My_classification_v2 LIMIT 10;

-----------------------------------------------------------
-- EVALUATION
-----------------------------------------------------------
CALL my_model_100k!SHOW_EVALUATION_METRICS();
CALL my_model_100k!SHOW_GLOBAL_EVALUATION_METRICS();
CALL my_model_100k!SHOW_CONFUSION_MATRIX();
CALL my_model_100k!SHOW_FEATURE_IMPORTANCE();

## 5. Resultats - Comparaison des deux approches

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

print("APPROCHE 1 : XGBoost Custom - jeu de TEST (20% = 33 007 clients)")
print("-" * 60)
y_pred_xgb = modele.predict(X_test)
report_xgb = classification_report(y_test, y_pred_xgb, output_dict=True)
print(f"Accuracy : {accuracy_score(y_test, y_pred_xgb):.1%}")
print(classification_report(y_test, y_pred_xgb, target_names=["Reste (0)", "Parti (1)"]))

print("\nAPPROCHE 2 : Classification Native - jeu EVAL interne (~1.5%)")
print("-" * 60)

native_precision_0 = 0.891
native_recall_0 = 0.958
native_f1_0 = 0.923
native_support_0 = 1849
native_precision_1 = 0.794
native_recall_1 = 0.578
native_f1_1 = 0.669
native_support_1 = 514
native_accuracy = (native_recall_0 * native_support_0 + native_recall_1 * native_support_1) / (native_support_0 + native_support_1)

print(f"Accuracy : {native_accuracy:.1%}")
print(f"{'':>14}precision    recall  f1-score   support")
print(f"   Reste (0)      {native_precision_0:.2f}      {native_recall_0:.2f}      {native_f1_0:.2f}      {native_support_0}")
print(f"   Parti (1)      {native_precision_1:.2f}      {native_recall_1:.2f}      {native_f1_1:.2f}       {native_support_1}")

print("\n\nTABLEAU COMPARATIF - PERFORMANCE")
print("-" * 60)

comparaison = pd.DataFrame({
    "Metrique": ["Accuracy", "Precision (Parti)", "Recall (Parti)", "F1 (Parti)", "Taille jeu de test"],
    "XGBoost Custom": [
        f"{accuracy_score(y_test, y_pred_xgb):.1%}",
        f"{report_xgb['1']['precision']:.1%}",
        f"{report_xgb['1']['recall']:.1%}",
        f"{report_xgb['1']['f1-score']:.1%}",
        "33 007 (20%)"
    ],
    "Classification Native": [
        f"{native_accuracy:.1%}",
        f"{native_precision_1:.1%}",
        f"{native_recall_1:.1%}",
        f"{native_f1_1:.1%}",
        "2 363 (~1.5%)"
    ]
})
print(comparaison.to_string(index=False))

print("\n\nFEATURE IMPORTANCE COMPAREE")
print("-" * 60)

fi_compare = pd.DataFrame({
    "Feature": ["NUMOFPRODUCTS", "ISACTIVEMEMBER", "AGE", "GEOGRAPHY_GERMANY",
                "GENDER_MALE", "BALANCE", "HASCRCARD", "CREDITSCORE",
                "ESTIMATEDSALARY", "TENURE", "GEOGRAPHY_SPAIN"],
    "XGBoost (rank)": ["1 (52.0%)", "2 (15.7%)", "3 (13.7%)", "4 (5.8%)",
                       "5 (4.8%)", "6 (3.1%)", "7 (1.3%)", "8 (1.0%)",
                       "9 (0.9%)", "10 (0.9%)", "11 (0.9%)"],
    "Natif (rank)": ["5 (9.1%)", "6 (6.2%)", "2 (17.3%)", "8 (5.0%)",
                     "9 (4.7%)", "1 (17.6%)", "10 (2.5%)", "3 (16.4%)",
                     "4 (14.7%)", "7 (5.7%)", "11 (0.8%)"]
})
print(fi_compare.to_string(index=False))

print("\nObservation : XGBoost concentre l'importance sur NUMOFPRODUCTS (52%)")
print("alors que la classification native repartit plus uniformement l'importance.")

print("\n\nTABLEAU COMPARATIF - COUT")
print("-" * 60)

cout = pd.DataFrame({
    "Poste": [
        "Compute (training)",
        "Compute (inference)",
        "Stockage modele",
        "Estimation totale"
    ],
    "XGBoost Custom": [
        "Notebook Service XS : ~1 credit/h",
        "Warehouse pour predict : ~1 credit",
        "Registry : negligeable",
        "~2-3 credits"
    ],
    "Classification Native": [
        "Warehouse ML_WH : ~10-15 credits",
        "Warehouse pour predict : ~2 credits",
        "Modele interne : negligeable",
        "~12-17 credits"
    ]
})
print(cout.to_string(index=False))

## 6. Options d'optimisation

### Pour XGBoost
| Piste | Description | Commande/Code | Impact attendu |
|-------|-------------|---------------|----------------|
| **Hyperparameter tuning** | GridSearch/RandomSearch sur max_depth, lr, n_estimators | `GridSearchCV(modele, param_grid, cv=5)` | +2-5% F1 |
| **Threshold tuning** | Ajuster le seuil de decision (ex: 0.3 au lieu de 0.5) | `proba = modele.predict_proba(X_test)[:,1]` | Meilleur recall |
| **SMOTE** | Surechantillonnage de la classe minoritaire | `from imblearn.over_sampling import SMOTE` | Meilleur recall |
| **Feature engineering** | Creer des interactions (AGE x BALANCE, TENURE x PRODUCTS) | Colonnes supplementaires | +1-3% |
| **Cross-validation** | 5-fold CV pour estimation plus fiable | `cross_val_score(modele, X, y, cv=5)` | Robustesse |

### Pour la Classification Native
| Piste | Description | Impact attendu |
|-------|-------------|----------------|
| **Plus de features** | Ajouter des colonnes au dataset source | Meilleure capture des patterns |
| **Nettoyage donnees** | Retirer les valeurs aberrantes | Moins de bruit |
| **Re-entrainer regulierement** | Snowflake Tasks pour automatiser | Modele toujours a jour |

### Analyse du recall faible (Parti)
Les deux approches ont un recall sous-optimal pour la classe "Parti" :
- XGBoost : 79% (rate 21% des churners)
- Natif : 58% (rate 42% des churners)

Cela signifie que des clients a risque passent entre les mailles. Solutions :
1. Baisser le seuil de classification (0.3 au lieu de 0.5)
2. Augmenter le `scale_pos_weight`
3. Ajouter des features temporelles (evolution du solde, frequence recente)

## 7. Pour aller plus loin

### Deploiement en production
| Etape | Commande | Description |
|-------|----------|-------------|
| Scorer un client | `SELECT CHURN_PREDICTOR!PREDICT(...)` | Prediction unitaire |
| Scorer en batch | `model_ref.run(df_new_clients)` | Scoring de masse |
| Planifier re-training | `CREATE TASK retrain_churn ...` | Automatisation |
| Monitoring | `CALL model!SHOW_EVALUATION_METRICS()` | Suivi dans le temps |

### Axes d'amelioration
| Axe | Description |
|-----|-------------|
| **Drift detection** | Comparer les distributions des features entre training et production |
| **A/B testing** | Comparer XGBoost vs Natif sur de vrais clients en parallele |
| **Segmentation** | Clustering des churners pour actions de retention personnalisees |
| **Explicabilite** | SHAP values pour expliquer chaque prediction au metier |
| **Pipeline MLOps** | Snowflake Tasks + Registry pour CI/CD du modele |
